**TTFT — Time to First Token (ms)**
The time from when the request is received to when the first token of the response is generated. This corresponds to the prefill (prompt processing) phase. TTFT directly controls perceived responsiveness — it is the "thinking delay" the user sees before output starts streaming.

**TPOT — Time Per Output Token (ms)**
The average time to generate each output token after the first. Calculated as `(total_latency - TTFT) / num_output_tokens`. TPOT reflects the average decode speed. Lower TPOT means faster overall generation.

**ITL — Inter-Token Latency (ms)**
The time between consecutive token emissions during streaming. While TPOT is an average, ITL captures the distribution — including jitter and spikes. High P99 ITL means the stream occasionally stalls, which users perceive as stuttering.

**E2EL — End-to-End Latency (ms)**
The total time from request submission to the last token. `E2EL = TTFT + (num_output_tokens x TPOT)`. This is the bottom-line metric for non-streaming use cases.

### How Each Parallelism Strategy Affects These Metrics

| Metric | Tensor Parallelism (TP) | Pipeline Parallelism (PP) | Data Parallelism (DP) |
|---|---|---|---|
| **TTFT** | Decreases (prefill compute split across GPUs) | Increases slightly (pipeline fill latency) | Unchanged per request, but lower under concurrency (less queueing) |
| **TPOT** | May decrease (faster per-layer compute) or increase (all-reduce overhead) | May increase (inter-stage communication) | Unchanged (each replica runs independently) |
| **ITL** | More consistent (even compute split) | Can be spiky (pipeline bubble stalls) | More consistent under load (less contention) |
| **Throughput** | Marginal improvement | Marginal improvement | Near-linear scaling with replica count |


In production, you combine strategies based on your constraints:

| Constraint | Strategy |
|---|---|
| Model fits on 1 GPU, need lower latency | TP (start with TP=2) |
| Model does not fit on 1 GPU | PP or TP (whichever the GPU count supports) |
| Model does not fit on 1 node | Multi-node PP (with intra-node TP) |
| Need higher throughput under load | DP (replicate model, split traffic) |
| Production large-model serving | TP x PP x DP combined |

The constraint: **TP x PP x DP = Total GPU count**

For example, on a 2-node cluster with 4 GPUs per node (8 GPUs total):

- TP=4 x PP=2 x DP=1 = 8 GPUs — One large-model replica across 2 nodes
- TP=2 x PP=1 x DP=4 = 8 GPUs — Four small-model replicas, each using TP=2
- TP=2 x PP=2 x DP=2 = 8 GPUs — Two replicas, each spanning 2 nodes with TP=2


| Config | Throughput | TTFT p50 | TTFT p99 | TPOT p50 | ITL p99 | Best For |
|---|---|---|---|---|---|---|
| TP=1 (baseline) | Baseline | Baseline | Baseline | Baseline | Baseline | Single-user, small models |
| TP=2 | Similar | Lower | Lower | Similar | Similar | Reducing latency |
| TP=4 | May drop | Lowest | Lowest | May increase | Similar | Large models, latency-critical |
| PP=2 | Similar | Higher | Higher | Higher | Higher | Fitting larger models |
| PP=4 | Similar | Highest | Highest | Higher | Highest | Very large models |
| TP=2 x PP=2 | Similar | Moderate | Moderate | Moderate | Moderate | Production large-model serving |
| DP=2 | ~2x | Same | Much lower* | Same | Much lower* | High-throughput serving |
| DP=2 x TP=2 | ~2x | Lower | Much lower* | Similar | Much lower* | Throughput + latency |

* *DP shows improvement under high concurrency , under low concurrency it doesnt show much improvement


### vLLM Serve Flags

| Flag | Description |
|---|---|
| `--tensor-parallel-size N` | Shard every layer across N GPUs |
| `--pipeline-parallel-size N` | Split layers into N sequential stages |
| `--data-parallel-size N` | Run N independent model replicas |
| `--distributed-executor-backend ray` | Use Ray for multi-node coordination |
| `--port 8000` | Port for the OpenAI-compatible API |
| `--gpu-memory-utilization 0.9` | Fraction of GPU memory vLLM can use for KV cache |

### vLLM Bench Serve Flags (Metrics Collection)

| Flag | Description |
|---|---|
| `--percentile-metrics ttft,tpot,itl,e2el` | Which metrics to report percentiles for |
| `--metric-percentiles 50,90,99` | Which percentiles to calculate (default: 99 only) |
| `--save-result` | Save results to file |
| `--output-json FILE` | Path for the JSON results file |
| `--request-rate N` | Requests per second to send |
| `--max-concurrency N` | Maximum concurrent requests |
| `--dataset-name random` | Use synthetic random prompts |
| `--random-input-len N` | Input token length for random dataset |
| `--random-output-len N` | Output token length for random dataset |
| `--goodput ttft:MS tpot:MS` | Define SLO thresholds for goodput calculation |
| `--ignore-eos` | Ignore EOS token to force exact output length |


1. **Prototype** — Single GPU, baseline metrics recorded. TTFT is okay, throughput is limited.
2. **Latency Wall** — TP splits layers across GPUs. TTFT drops. Users see faster first responses.
3. **Memory Wall** — PP splits layer groups across nodes. Models that cannot fit on one machine now run.
4. **Throughput Wall** — DP replicates the model. Throughput scales linearly. P99 latencies drop under load.
5. **Production** — Combine TP x PP x DP to serve large models at scale with acceptable latency and throughput.